# 02 - Vision Transformers

**Architecture block 2a.** Four transformer families behind one interface:
DINOv2-S (self-supervised, the primary backbone), ViT-B/16 (supervised),
Swin-T (hierarchical) and DeiT-S (distilled).

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent / "src"))

import cropforecast
from cropforecast.config import load_config, ensure_dirs, set_seed, Device
cfg = load_config(Path.cwd().parent / "configs" / "default.yaml")
ensure_dirs(cfg); set_seed(cfg.project.seed)
device = Device.auto(cfg.training.amp)
print("device:", device)

In [ ]:
import torch
from cropforecast.models.backbones import SPECS, load_backbone
for k, s in SPECS.items():
    print(f"{k:8s} {s.hf_id:42s} dim={s.dim:4d} pool={s.pool:5s} {s.family}")

### Patch embedding and self-attention

The diagram's steps 3-4: split the image into patches, embed them, add
positional encoding, then run L encoder blocks of multi-head self-attention.

In [ ]:
bb = load_backbone("dinov2", frozen=True).to(device.name).eval()
x = torch.randn(2, 3, 224, 224, device=device.name)
with torch.no_grad():
    out = bb(x)
print("pooled (CLS) :", tuple(out["pooled"].shape))
print("patch tokens :", tuple(out["tokens"].shape), "-> 1 CLS + 16x16 patches")
print("frozen params:", sum(p.numel() for p in bb.parameters())/1e6, "M")

### A real leaf through the backbone

In [ ]:
import numpy as np, pandas as pd
from PIL import Image
from transformers import AutoImageProcessor
obs = pd.read_parquet(Path(cfg.paths.processed) / "observations.parquet")
row = obs[obs.class_name == "Tomato___Late_blight"].iloc[0]
img = Image.open(row.image_path).convert("RGB").resize((224, 224))

proc = AutoImageProcessor.from_pretrained(SPECS["dinov2"].hf_id)
mean = torch.tensor(proc.image_mean, device=device.name).view(1,3,1,1)
std  = torch.tensor(proc.image_std,  device=device.name).view(1,3,1,1)
t = torch.from_numpy(np.array(img, dtype=np.uint8)).permute(2,0,1)[None].to(device.name).float()/255
t = (t - mean)/std
with torch.no_grad():
    emb = bb(t)["pooled"]
print(row.class_name, "->", tuple(emb.shape))
display(img)

### Attention rollout: where is the transformer looking?

In [ ]:
from cropforecast.explain.vision import attention_rollout, overlay
import matplotlib.pyplot as plt
bb_attn = load_backbone("dinov2", frozen=True, output_attentions=True).to(device.name).eval()
roll = attention_rollout(bb_attn, t)
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(np.array(img)); ax[0].set_title("input"); ax[0].axis("off")
ax[1].imshow(overlay(np.array(img), roll)); ax[1].set_title("attention rollout"); ax[1].axis("off")
plt.show()

### Benchmark results

Run `python scripts/03_benchmark_backbones.py` first.

In [ ]:
bench = Path(cfg.paths.reports) / "stage3_backbone_benchmark.csv"
pd.read_csv(bench) if bench.exists() else print("Run scripts/03_benchmark_backbones.py")